# Bài 2.10 · Phân cụm trên dữ liệu thật

**Câu hỏi của buổi này:** không có nhãn thì máy chia nhóm thế nào, chọn số nhóm ra sao, và vì sao cùng một dữ liệu mà mỗi thuật toán chia một kiểu?

| Mục | Sẽ thấy |
|---|---|
| 1. k-means nén màu một bức ảnh | ảnh chỉ còn 2, 4, 8, 16, 64 màu |
| 2. Vòng Lloyd từng bước | tâm cụm di chuyển qua bốn vòng |
| 3. Chọn k và chuẩn hóa | inertia, silhouette theo k; chuẩn hóa thay đổi kết quả thế nào |
| 4. DBSCAN trên bản đồ | các vùng dân cư đông của California, điểm nhiễu |
| 5. Hierarchical | dendrogram của 40 mẫu rượu |
| 6. So các thuật toán | bảng ARI, NMI, silhouette |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **Internet** trong *Session options*; CPU là đủ, chạy hết khoảng 1 phút. Bài giảng: [ai-canuc80k · bài 2.10](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/2.10).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, sklearn
from sklearn.datasets import load_sample_image, load_wine, fetch_california_housing
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from scipy.cluster.hierarchy import linkage, dendrogram

np.random.seed(0)
table(pd.DataFrame([('scikit-learn', sklearn.__version__), ('numpy', np.__version__)], columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. k-means nén màu một bức ảnh

Mỗi pixel là một điểm 3 chiều (R, G, B). Chia 273 000 pixel thành $k$ cụm, rồi tô mỗi pixel bằng màu tâm cụm của nó: ảnh chỉ còn $k$ màu. **Inertia** là tổng bình phương khoảng cách từ mỗi pixel tới tâm cụm của nó.

In [ ]:
img = load_sample_image('china.jpg') / 255.0
H, W, _ = img.shape
P = img.reshape(-1, 3)
sample = P[np.random.default_rng(0).choice(len(P), 20000, replace=False)]
fig, axes = plt.subplots(2, 3, figsize=(15, 7.4))
axes[0, 0].imshow(img); axes[0, 0].set_title(f'gốc: {len(np.unique((P * 255).astype(int), axis=0)):,} màu'); axes[0, 0].axis('off')
rows = []
for ax, k in zip(axes.flat[1:], [2, 4, 8, 16, 64]):
    km = KMeans(k, n_init=3, random_state=0).fit(sample)
    lab = km.predict(P)
    ax.imshow(km.cluster_centers_[lab].reshape(H, W, 3)); ax.set_title(f'k = {k} màu'); ax.axis('off')
    rows.append((k, km.inertia_ / len(sample), np.mean((km.cluster_centers_[lab] - P) ** 2)))
plt.tight_layout(); plt.show()
quant = pd.DataFrame(rows, columns=['k', 'inertia mỗi pixel (mẫu)', 'sai số màu trung bình (cả ảnh)'])
table(quant, f'Ảnh {W} × {H} = {H * W:,} pixel', fmt={'inertia mỗi pixel (mẫu)': '{:.4f}', 'sai số màu trung bình (cả ảnh)': '{:.5f}'})
check('k càng lớn, inertia càng nhỏ', (np.diff(quant.iloc[:, 1]) < 0).all())
note(f"16 màu đã giữ được hình và phần lớn sắc độ (sai số {quant.iloc[3, 2]:.4f}), chỉ bầu trời còn thành từng mảng; 64 màu thì gần như không khác ảnh gốc. Với 16 màu, mỗi pixel chỉ cần 4 bit thay vì 24 bit.")

## 2. Vòng Lloyd từng bước

k-means lặp hai bước: **gán** mỗi điểm cho tâm gần nhất, rồi **cập nhật** mỗi tâm thành trung bình các điểm của nó. Dữ liệu là bộ **Wine** (178 mẫu rượu, 13 chỉ số hóa học, 3 giống nho), chuẩn hóa rồi chiếu PCA xuống 2 chiều để nhìn.

In [ ]:
wine = load_wine()
Xw, yw = wine.data, wine.target
Z = PCA(2, random_state=0).fit_transform(StandardScaler().fit_transform(Xw))
cent = Z[[0, 1, 2]].copy()                           # cố ý khởi tạo tệ: ba mẫu đầu, cùng một giống
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
moves = []
for it, ax in enumerate(axes):
    lab = np.argmin(((Z[:, None] - cent[None]) ** 2).sum(-1), axis=1)
    inertia = ((Z - cent[lab]) ** 2).sum()
    ax.scatter(Z[:, 0], Z[:, 1], c=[ORDER[l] for l in lab], s=14, alpha=.7)
    ax.scatter(cent[:, 0], cent[:, 1], c='black', marker='X', s=160)
    new = np.array([Z[lab == j].mean(0) for j in range(3)])
    for a, b in zip(cent, new): ax.annotate('', b, a, arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    ax.set_title(f'vòng {it + 1}: inertia {inertia:.0f}'); ax.set_xticks([]); ax.set_yticks([])
    moves.append(np.abs(new - cent).max()); cent = new
plt.tight_layout(); plt.show()
check('tâm cụm dịch chuyển ít dần', moves[-1] < moves[0])
note(f'Dấu X là tâm, mũi tên là chỗ tâm sẽ đến sau bước cập nhật. Tâm dịch nhiều nhất {moves[0]:.2f} ở vòng 1, còn {moves[-1]:.3f} ở vòng 4.')

## 3. Chọn k và chuẩn hóa

**Inertia** luôn giảm khi tăng $k$, nên người ta tìm chỗ "khuỷu tay". **Silhouette** (từ −1 đến 1) so mỗi điểm gần cụm của nó hơn cụm bên cạnh bao nhiêu; cao là tốt. Bộ Wine có 3 giống nho thật, dùng để kiểm (không dùng để phân cụm).

Cột `proline` của Wine có giá trị cỡ hàng trăm, các cột khác cỡ 0.1–10. Không chuẩn hóa thì khoảng cách gần như chỉ do một cột ấy quyết định.

In [ ]:
Xs = StandardScaler().fit_transform(Xw)
rows = []
for k in range(2, 9):
    km = KMeans(k, n_init=10, random_state=0).fit(Xs)
    rows.append((k, km.inertia_, silhouette_score(Xs, km.labels_), adjusted_rand_score(yw, km.labels_)))
ks = pd.DataFrame(rows, columns=['k', 'inertia', 'silhouette', 'ARI với giống nho'])
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
axes[0].plot(ks['k'], ks['inertia'], marker='o', color=C['blue']); axes[0].set(xlabel='k', title='inertia (càng tăng k càng giảm)')
axes[1].plot(ks['k'], ks['silhouette'], marker='o', color=C['clay']); axes[1].set(xlabel='k', title='silhouette (cao là tốt)')
for ax in axes: ax.axvline(3, color='#999', ls=':')
plt.tight_layout(); plt.show()
table(ks, 'Wine đã chuẩn hóa, k-means theo k', fmt={'inertia': '{:.0f}', 'silhouette': '{:.3f}', 'ARI với giống nho': '{:.3f}'}, best='silhouette')

raw = KMeans(3, n_init=10, random_state=0).fit(Xw)
sc3 = KMeans(3, n_init=10, random_state=0).fit(Xs)
scale = pd.DataFrame([('không chuẩn hóa', adjusted_rand_score(yw, raw.labels_)), ('chuẩn hóa từng cột', adjusted_rand_score(yw, sc3.labels_))], columns=['k-means k = 3', 'ARI với giống nho'])
table(scale, f'Độ lệch chuẩn cột proline: {Xw[:, 12].std():.0f}; các cột khác: {np.median(Xw[:, :12].std(0)):.2f} (trung vị)', fmt={'ARI với giống nho': '{:.3f}'}, best='ARI với giống nho')
best_k = int(ks.loc[ks['silhouette'].idxmax(), 'k'])
note(f"Silhouette cao nhất ở <b>k = {best_k}</b>. Chuẩn hóa đưa ARI từ {scale.iloc[0, 1]:.2f} lên <b>{scale.iloc[1, 1]:.2f}</b>: thiếu bước này, k-means chỉ chia theo proline.")

## 4. DBSCAN trên bản đồ

**DBSCAN** không cần biết trước số cụm: một điểm là **lõi** nếu trong bán kính `eps` có ít nhất `min_samples` điểm; các lõi nối nhau thành cụm; điểm không gần lõi nào là **nhiễu**. Dữ liệu: tọa độ 20 640 khu dân cư (block group) ở California. Khoảng cách tính theo km trên mặt cầu (haversine).

In [ ]:
with quiet():
    cal = fetch_california_housing(as_frame=True).frame
geo = np.radians(cal[['Latitude', 'Longitude']].to_numpy())
R = 6371.0
t = time.time()
db = DBSCAN(eps=8 / R, min_samples=150, metric='haversine', algorithm='ball_tree').fit(geo)
lab = db.labels_
n_clu = lab.max() + 1
km_geo = KMeans(n_clu, n_init=10, random_state=0).fit_predict(cal[['Latitude', 'Longitude']])
fig, axes = plt.subplots(1, 2, figsize=(14, 6.2))
for ax, L, title in [(axes[0], lab, f'DBSCAN (eps 8 km, min_samples 150): {n_clu} cụm, nhiễu màu xám'), (axes[1], km_geo, f'k-means với cùng k = {n_clu}')]:
    noise = L == -1
    ax.scatter(cal['Longitude'][noise], cal['Latitude'][noise], s=1, c='#ccc')
    ax.scatter(cal['Longitude'][~noise], cal['Latitude'][~noise], s=2, c=L[~noise], cmap='tab20')
    ax.set_title(title, fontsize=11); ax.set_xlabel('kinh độ'); ax.set_ylabel('vĩ độ'); ax.set_aspect(1.15)
plt.tight_layout(); plt.show()
sizes = pd.Series(lab[lab >= 0]).value_counts()
centers = cal.groupby(lab)[['Latitude', 'Longitude']].mean().drop(-1, errors='ignore').loc[sizes.index[:5]]
table(pd.DataFrame({'cụm': sizes.index[:5], 'số khu': sizes.values[:5], 'vĩ độ tâm': centers['Latitude'].values, 'kinh độ tâm': centers['Longitude'].values}),
      f'Năm cụm DBSCAN lớn nhất ({time.time() - t:.0f} giây)', fmt={'vĩ độ tâm': '{:.2f}', 'kinh độ tâm': '{:.2f}'})
check(f'{(lab == -1).mean():.0%} khu bị coi là nhiễu (vùng thưa dân)', 0 < (lab == -1).mean() < 1)
note(f'DBSCAN tự tìm {n_clu} vùng đông dân (cụm lớn nhất quanh vĩ độ {centers.iloc[0, 0]:.1f}, kinh độ {centers.iloc[0, 1]:.1f} là vùng Los Angeles) và bỏ các khu thưa thành nhiễu. '
     f'k-means với cùng số cụm buộc mọi khu vào một cụm, kể cả vùng núi, sa mạc.')

## 5. Hierarchical clustering

Bắt đầu mỗi điểm một cụm, lặp lại việc gộp hai cụm gần nhau nhất (Ward: gộp cặp làm tăng tổng phương sai ít nhất). **Dendrogram** ghi lại thứ tự gộp; cắt ngang ở độ cao nào thì được bấy nhiêu cụm. Vẽ cho 40 mẫu Wine, nhãn dưới là giống nho thật.

In [ ]:
pick = np.concatenate([np.flatnonzero(yw == c)[:14 if c < 2 else 12] for c in range(3)])
Lk = linkage(Xs[pick], 'ward')
fig, ax = plt.subplots(figsize=(15, 4.4))
d = dendrogram(Lk, labels=[f'giống {yw[i]}' for i in pick], leaf_font_size=8, ax=ax, color_threshold=Lk[-3, 2] + 0.01, above_threshold_color='#999')
cols = {0: C['blue'], 1: C['clay'], 2: C['green']}
for t_ in ax.get_xticklabels(): t_.set_color(cols[int(t_.get_text()[-1])])
ax.axhline(Lk[-3, 2] + 0.5, color='#333', ls='--', lw=1); ax.text(2, Lk[-3, 2] + 0.9, 'cắt ở đây: 3 cụm', fontsize=10)
ax.set_title('Dendrogram Ward của 40 mẫu Wine (nhãn tô màu theo giống thật)'); ax.set_ylabel('khoảng cách gộp')
plt.tight_layout(); plt.show()

## 6. So các thuật toán

Cùng Wine đã chuẩn hóa, cùng 3 cụm (DBSCAN tự quyết). **ARI** và **NMI** so với giống nho thật (chỉ dùng để chấm); **silhouette** chỉ dùng chính dữ liệu.

In [ ]:
algos = {'k-means': KMeans(3, n_init=10, random_state=0), 'hierarchical (Ward)': AgglomerativeClustering(3),
         'Gaussian mixture': GaussianMixture(3, random_state=0), 'DBSCAN (eps 2.3)': DBSCAN(eps=2.3, min_samples=5)}
rows = []
for n, a in algos.items():
    L = a.fit_predict(Xs)
    m = L >= 0
    rows.append((n, len(set(L[m])), (~m).mean(), adjusted_rand_score(yw, L), normalized_mutual_info_score(yw, L), silhouette_score(Xs[m], L[m]) if len(set(L[m])) > 1 else np.nan))
cmp = pd.DataFrame(rows, columns=['thuật toán', 'số cụm', 'tỉ lệ nhiễu', 'ARI', 'NMI', 'silhouette'])
table(cmp, 'Wine đã chuẩn hóa: bốn cách chia', fmt={'tỉ lệ nhiễu': '{:.0%}', 'ARI': '{:.3f}', 'NMI': '{:.3f}', 'silhouette': '{:.3f}'}, best='ARI')
best = cmp.loc[cmp['ARI'].idxmax()]
note(f"Trên bộ này, <b>{best['thuật toán']}</b> khớp giống nho nhất (ARI {best['ARI']:.3f}). DBSCAN hợp với dữ liệu có vùng đặc, vùng thưa rõ (như bản đồ ở mục 4) hơn là 13 chỉ số hóa học trải đều.")

## 7. Tóm tắt

In [ ]:
summary = [('nén ảnh: sai số màu với 16 màu', f'{quant.iloc[3, 2]:.5f}'),
           ('Wine: k có silhouette cao nhất', f'{best_k}'),
           ('Wine k-means: ARI không / có chuẩn hóa', f'{scale.iloc[0, 1]:.2f} / {scale.iloc[1, 1]:.2f}'),
           ('California: số cụm DBSCAN / tỉ lệ nhiễu', f'{n_clu} / {(lab == -1).mean():.0%}'),
           ('Wine: ARI tốt nhất', f"{best['thuật toán']} {best['ARI']:.3f}")]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** ở mục 1 dùng ảnh `flower.jpg`; ở mục 4 đổi `eps` thành 4 km hay 20 km và `min_samples` thành 50; ở mục 2 khởi tạo bằng `k-means++` rồi đếm số vòng tới khi hội tụ.